# Full pipeline — Ingesta through Classification & Routing, end-to-end

Exercises the whole system exactly as production runs it: one `POST /pipeline/ingest`
call chains automatically through Stage 1 (Ingesta) → Stage 2 (extraction) → Stage 3
(Refinement & Enrichment) → Stage 4 (Classification & Routing) inside
`PipelineService._run()` — there is no separate trigger for later stages, and no
shortcut through the coordinators or nodes directly. Every call below goes through
`TestClient` and the real FastAPI routes, exactly as `playground/stage1/pipeline_end_to_end.ipynb`
and `playground/stage3/enrichment_end_to_end.ipynb` already do for their own stages —
this notebook is their direct continuation through the newly-completed Stage 4.

Uses the **real** production `Container`: real SQL repos against
`Settings.DATABASE_URL`, real MarkItDown/OCR extraction, real text cleaning/entity
extraction, and — unlike the Stage 3 notebook — the **real** local models for every
Stage 4 agent too: the real Phi-4-mini primary classifier, the real BETO Second
Opinion Agent (`models/bert_tunning_beto_v2`, `second_opinion_enabled=true` by
default), and the real Gemma 4 LLM Judge. Only node3's legitimacy check is mocked
(`set_legitimacy()` below), so acceptance into the pipeline stays deterministic for
the demo regardless of what the real model would decide about a given sample PDF —
everything downstream of that gate runs for real.

> **Heads up**: section 1 below deletes and recreates `data/classiflow.db` at the
> start of every run — same as the Stage 1 and Stage 3 notebooks — so each run starts
> from a clean slate. No leftover jobs, no exact-duplicate rejections at node4 from a
> previous run's identical file bytes. Back up `data/classiflow.db` first if you want
> to keep a previous run's data.

> **Also heads up**: this run loads five separate local models into memory/VRAM in
> sequence over the course of the notebook (Phi-4-mini for node2/node3, BETO for the
> Second Opinion Agent, Gemma 4 for the LLM Judge) — expect the first classification
> cell to take noticeably longer than Stage 1/3's own notebooks while models load.
> `PipelineService._run()`'s own `unload_slm()` call releases the GGUF models' GPU VRAM
> after each job finishes, so this is a per-job cost, not a cumulative leak across the
> jobs this notebook submits.

## 1 — App setup: the real Container, JWT auth, a blanked database

In [1]:
import shutil
from pathlib import Path

from fastapi.testclient import TestClient
from sqlalchemy import select
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

import classiflow
from classiflow.api.app import create_app
from classiflow.database.base import Base
from classiflow.database.models import AllowedUser, ClassificationRecord, EnrichedRecord, Job
from classiflow.injections.production import Container
from classiflow.services.auth import encode_token
from classiflow.settings import Settings

Settings.JWT_SECRET_KEY = "playground-secret-key-not-for-prod-use-only-demo"

# Settings.DATABASE_URL defaults to a *relative* path, which breaks with "unable to
# open database file" when the kernel's cwd isn't the repo root -- anchor it to the
# actual package location instead, same reasoning as the Stage 1/3 notebooks.
_project_root = Path(classiflow.__file__).parents[2]
_db_path = _project_root / "data" / "classiflow.db"

# Blank the database before anything opens a connection to it -- every run starts
# from a clean slate, so node4's exact-duplicate check never trips on a previous
# run's identical file bytes, and the classification/review-queue inspections below
# only ever show this run's own jobs.
for _stale in (_db_path, _db_path.with_suffix(".db-wal"), _db_path.with_suffix(".db-shm")):
    if _stale.exists():
        _stale.unlink()
print(f"reset database at {_db_path}")

Settings.DATABASE_URL = f"sqlite+aiosqlite:///{_db_path.as_posix()}"

# The DB reset above only clears rows -- it does not touch storage/documents/, which
# RoutingNode/DocumentStorage write real files into (staging/, classified/<label>/,
# review/human_review/). Those files persist across notebook runs (each job writes a
# fresh uuid-prefixed filename, so nothing here ever overwrites or dedupes a previous
# run's output) and were the actual cause of an earlier "3 files ingested but 4
# classification files on disk" mismatch -- section 8's file count included leftovers
# from a prior run, not an extra file this run produced. Wipe it here too so section
# 8 always reflects only this run's own jobs.
_storage_root = Path(Settings.document_storage_root)
if _storage_root.exists():
    shutil.rmtree(_storage_root)
print(f"reset storage at {_storage_root}")

container = Container()
container.wire(packages=["classiflow"])

engine = create_async_engine(Settings.DATABASE_URL, echo=False)
session_factory = async_sessionmaker(engine, expire_on_commit=False)


async def _create_tables() -> None:
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)


async def _seed_user(email: str) -> None:
    async with session_factory() as session:
        existing = await session.execute(select(AllowedUser).where(AllowedUser.email == email))
        if existing.scalar_one_or_none() is None:
            session.add(AllowedUser(email=email, is_active=True, is_blocked=False))
            await session.commit()


await _create_tables()
_EMAIL = "leonardo.heis@gmail.com"
await _seed_user(_EMAIL)

client = TestClient(create_app())
auth_headers = {"Authorization": f"Bearer {encode_token(_EMAIL)}"}

print(f"logged in as {_EMAIL}")
print(f"writing to {_db_path}")

c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\leona\source\repos\Trabajo-Integrador\src\classiflow\api\dependencies.py:347: DIWiringWarning: @inject is not required here
  def get_coordinator(


reset database at C:\Users\leona\source\repos\Trabajo-Integrador\data\classiflow.db
reset storage at C:\Users\leona\source\repos\Trabajo-Integrador\storage\documents
logged in as leonardo.heis@gmail.com
writing to C:\Users\leona\source\repos\Trabajo-Integrador\data\classiflow.db


## 2 — Real sample PDFs, and the one mock (node3's legitimacy check)

`set_legitimacy(is_legitimate=True)` guarantees node3 accepts the document, so it
always reaches Stage 3/4 regardless of what the real SLM would decide about a given
sample. This is the only mock in this notebook — node2's format check, all of
Stage 3's enrichment, and every Stage 4 classification agent (primary classifier,
Second Opinion, foreign-municipality detector, smells/risk, confidence gate, LLM
Judge, Routing) run against their real local models.

The mock overrides the `Container`'s own `node3_content_chain` provider directly
(`container.node3_content_chain.override(...)`), the same technique the Stage 1/3
notebooks use — the chain a node actually receives comes from
`injections/production.py`'s own import of `get_llm_langchain`, a *different* name
binding than `node3_content_validation.py`'s own import, so monkeypatching one
wouldn't affect the other. Overriding the provider is the reliable way to mock one
piece of a real, fully-wired `Container`.

In [2]:
from pathlib import Path

from dependency_injector import providers

import classiflow
from classiflow.ingesta.llm_provider import MockLlm
from classiflow.ingesta.prompts import build_content_chain

_SAMPLES_DIR = Path(classiflow.__file__).parent / "playground" / "samples"

_SLM_LEGITIMATE = '{"is_legitimate": true, "confidence": 0.92, "reasoning": "official doc"}'
_SLM_NOT_LEGITIMATE = '{"is_legitimate": false, "confidence": 0.9, "reasoning": "n/a"}'


def set_legitimacy(*, is_legitimate: bool) -> None:
    response = _SLM_LEGITIMATE if is_legitimate else _SLM_NOT_LEGITIMATE
    container.node3_content_chain.override(
        providers.Object(build_content_chain(MockLlm(response=response)))
    )


def upload(filename: str) -> dict[str, tuple[str, bytes, str]]:
    file_bytes = (_SAMPLES_DIR / filename).read_bytes()
    return {"file": (filename, file_bytes, "application/pdf")}


# filename -> expected DocumentCategory value, taken from the labeled corpus this
# file was pulled from (each source folder name IS the ground-truth label). 2-3
# files per category, covering all 9 categories the labeled corpus has samples for
# -- compendios_de_boletines is excluded, the corpus has zero examples of it.
_SAMPLE_FILES: dict[str, str] = {
    "ordenanza_10000_2019.pdf": "ordenanzas",
    "convenio_2_2013.pdf": "convenios",
    "convenio_10_2013.pdf": "convenios",
    "convenio_229_2021.pdf": "convenios",
    "boletin_2056_2026.pdf": "boletines",
    "boletin_1000_2019.pdf": "boletines",
    "declaracion_2501_1991.pdf": "declaraciones_concejo_municipal",
    "declaracion_4918_2009.pdf": "declaraciones_concejo_municipal",
    "decreto_ordenanza_1182_1976.pdf": "decreto_ordenanzas",
    "decreto_ordenanza_1314_1980.pdf": "decreto_ordenanzas",
    "decreto_1000_2008.pdf": "decretos",
    "decreto_1001_2016.pdf": "decretos",
    "decreto_cm_1016_2025.pdf": "decretos_concejo_municipal",
    "decreto_cm_10554_1995.pdf": "decretos_concejo_municipal",
    "resolucion_100_2020.pdf": "resoluciones",
    "resolucion_102_2021.pdf": "resoluciones",
    "resolucion_cm_16879_2024.pdf": "resoluciones_concejo_municipal",
    "resolucion_cm_197_2024.pdf": "resoluciones_concejo_municipal",
}
for _name in _SAMPLE_FILES:
    _size = (_SAMPLES_DIR / _name).stat().st_size
    print(f"{_name}: {_size:,} bytes")

ordenanza_10000_2019.pdf: 121,870 bytes
convenio_2_2013.pdf: 121,098 bytes
convenio_10_2013.pdf: 116,975 bytes
convenio_229_2021.pdf: 90,081 bytes
boletin_2056_2026.pdf: 343,177 bytes
boletin_1000_2019.pdf: 187,913 bytes
declaracion_2501_1991.pdf: 107,387 bytes
declaracion_4918_2009.pdf: 157,129 bytes
decreto_ordenanza_1182_1976.pdf: 507,028 bytes
decreto_ordenanza_1314_1980.pdf: 149,773 bytes
decreto_1000_2008.pdf: 80,425 bytes
decreto_1001_2016.pdf: 181,925 bytes
decreto_cm_1016_2025.pdf: 198,603 bytes
decreto_cm_10554_1995.pdf: 249,004 bytes
resolucion_100_2020.pdf: 42,596 bytes
resolucion_102_2021.pdf: 33,945 bytes
resolucion_cm_16879_2024.pdf: 425,124 bytes
resolucion_cm_197_2024.pdf: 168,616 bytes


## 3 — Ingest all sample documents

`TestClient` runs FastAPI's background tasks synchronously as part of the call, and
`PipelineService._run()` chains straight through `_run_enrichment()` →
`_run_classification()` once a job is accepted — so by the time each `client.post(...)`
below returns, Stage 1 through Stage 4 have *all* already finished for that document,
no extra waiting needed. This ingests every file in `_SAMPLE_FILES` (2-3 per category,
covering all 9 categories the labeled corpus has ground truth for) -- expect this cell
to take a while, each job runs the full 5-model pipeline.

In [3]:
from classiflow.classification.exceptions import ClassificationError

set_legitimacy(is_legitimate=True)


# A raised ClassificationError (e.g. the primary classifier's JSON-escaping bug --
# see section 9) propagates straight out of PipelineService._run's background task,
# uncaught -- and TestClient runs background tasks synchronously inside the request,
# so it surfaces right here as an exception from client.post(). A helper function
# (rather than an inline try/except in the loop body) keeps the per-file catch out
# of the loop's hot path.
def _ingest_one(name: str) -> str | None:
    try:
        response = client.post("/pipeline/ingest", files=upload(name), headers=auth_headers)
    except ClassificationError as exc:
        print(f"{name}: INGEST CRASHED -- {type(exc).__name__}: {exc}")
        ingest_errors[name] = f"{type(exc).__name__}: {exc}"
        return None
    job_id: str = response.json()["jobId"]
    print(f"{name}: status={response.status_code} job_id={job_id}")
    return job_id


job_ids: dict[str, str] = {}
ingest_errors: dict[str, str] = {}
for _name in _SAMPLE_FILES:
    _job_id = _ingest_one(_name)
    if _job_id is not None:
        job_ids[_name] = _job_id

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 8191 MiB):
  Device 0: NVIDIA RTX A4000 Laptop GPU, compute capability 8.6, VMM: yes, VRAM: 8191 MiB
llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10173.88it/s]
llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
2026-08-21 15:09:05.853 | INFO     | classiflow.services.audit.service:record:37 - audit | job=72f4b5bc-98ae-420b-82cd-5a8c117798c6 node=node1_file_reception event=passed
2026-08-21 15:09:05.865 | INF

ordenanza_10000_2019.pdf: status=202 job_id=72f4b5bc-98ae-420b-82cd-5a8c117798c6


llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
2026-08-21 15:09:22.705 | INFO     | classiflow.services.audit.service:record:37 - audit | job=ae5279c5-43a1-49f5-9655-310868a3bec1 node=node1_file_reception event=passed
2026-08-21 15:09:22.715 | INFO     | classiflow.services.audit.service:record:37 - audit | job=ae5279c5-43a1-49f5-9655-310868a3bec1 node=node2_format_validation event=passed
2026-08-21 15:09:23.102 | INFO     | classiflow.services.audit.service:recor

convenio_2_2013.pdf: status=202 job_id=ae5279c5-43a1-49f5-9655-310868a3bec1


llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
2026-08-21 15:09:38.657 | INFO     | classiflow.services.audit.service:record:37 - audit | job=cf04d7f9-72c1-4061-bb15-5f3565ebfbdf node=node1_file_reception event=passed
2026-08-21 15:09:38.664 | INFO     | classiflow.services.audit.service:record:37 - audit | job=cf04d7f9-72c1-4061-bb15-5f3565ebfbdf node=node2_format_validation event=passed
2026-08-21 15:09:39.041 | INFO     | classiflow.services.audit.service:recor

...

resolucion_102_2021.pdf: status=202 job_id=dc7b2862-d9a0-428e-8b15-a380f3f4c682


llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
2026-08-21 15:14:00.422 | INFO     | classiflow.services.audit.service:record:37 - audit | job=b47a3d6d-e524-4738-9173-35cbb329e2ab node=node1_file_reception event=passed
2026-08-21 15:14:00.432 | INFO     | classiflow.services.audit.service:record:37 - audit | job=b47a3d6d-e524-4738-9173-35cbb329e2ab node=node2_format_validation event=passed
2026-08-21 15:14:39.936 | INFO     | classiflow.services.audit.service:recor

resolucion_cm_16879_2024.pdf: status=202 job_id=b47a3d6d-e524-4738-9173-35cbb329e2ab


llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
2026-08-21 15:14:56.355 | INFO     | classiflow.services.audit.service:record:37 - audit | job=8b0222c1-0997-4207-97b9-bc20fbfeb582 node=node1_file_reception event=passed
2026-08-21 15:14:56.355 | INFO     | classiflow.services.audit.service:record:37 - audit | job=8b0222c1-0997-4207-97b9-bc20fbfeb582 node=node2_format_validation event=passed
2026-08-21 15:15:24.759 | INFO     | classiflow.services.audit.service:recor

resolucion_cm_197_2024.pdf: status=202 job_id=8b0222c1-0997-4207-97b9-bc20fbfeb582


## 4 — Inspect each job's final `Job` status

In [4]:
async def _find_job(job_id: str) -> Job | None:
    async with session_factory() as session:
        result = await session.execute(select(Job).where(Job.job_id == job_id))
        return result.scalar_one_or_none()


for _name, _job_id in job_ids.items():
    job = await _find_job(_job_id)
    assert job is not None
    print(f"{_name}")
    print(f"  status               : {job.status!r}")
    print(f"  failed_at_node       : {job.failed_at_node!r}")
    print(f"  rejection_reason     : {job.rejection_reason!r}")
    print(f"  review_action_needed : {job.review_action_needed!r}")

ordenanza_10000_2019.pdf
  status               : 'accepted'
  failed_at_node       : None
  rejection_reason     : None
  review_action_needed : None
convenio_2_2013.pdf
  status               : 'accepted'
  failed_at_node       : None
  rejection_reason     : None
  review_action_needed : None
convenio_10_2013.pdf
  status               : 'accepted'
  failed_at_node       : None
  rejection_reason     : None
  review_action_needed : None
convenio_229_2021.pdf
  status               : 'accepted'
  failed_at_node       : None
  rejection_reason     : None
  review_action_needed : None
boletin_2056_2026.pdf
  status               : 'accepted'
  failed_at_node       : None
  rejection_reason     : None
  review_action_needed : None
boletin_1000_2019.pdf
  status               : 'accepted'
  failed_at_node       : None
  rejection_reason     : None
  review_action_needed : None
declaracion_2501_1991.pdf
  status               : 'accepted'
  failed_at_node       : None
  rejection_reason  

## 5 — Inspect the `EnrichedRecord` Stage 3 wrote

`cleaned_text` is the exact text Stage 4's primary classifier and LLM Judge see —
compare it against a raw PDF to see the text cleaner's repeated-line stripping and
noise removal in action.

In [5]:
async def _find_enriched_record(job_id: str) -> EnrichedRecord | None:
    async with session_factory() as session:
        result = await session.execute(
            select(EnrichedRecord).where(EnrichedRecord.job_id == job_id)
        )
        return result.scalar_one_or_none()


for _name, _job_id in job_ids.items():
    record = await _find_enriched_record(_job_id)
    print(f"{_name}")
    if record is None:
        print("  no EnrichedRecord -- job wasn't accepted, or enrichment failed")
        continue
    print(f"  cleaned_text ({len(record.cleaned_text)} chars): {record.cleaned_text[:200]!r}...")
    print(f"  entities: {record.entities}")
    print(f"  metadata: {record.metadata_}")

ordenanza_10000_2019.pdf
  cleaned_text (5235 chars): 'Concejo Municipal\nde IQsario\nLA MUNICIPALIDAD DE ROSARIO HA SANCIONADO LA SIGUIENTE\n(N0 10.000)\nConcejo Municipal\nVuestra Comisión de Salud y Acción Social ha considerado el\nproyecto de Ordenanza pres'...
  entities: {'doc_type_hint': None, 'number': 'N0 10.000', 'year': 2019, 'issuing_body': 'Concejo Municipal de IQsario (Municipalidad de Rosario)', 'signatories': ['Dr. Alejandro Rosello'], 'article_count': 4}
  metadata: {'source': 'manual_upload', 'filename': 'ordenanza_10000_2019.pdf', 'language': 'es', 'sha256': 'ac6486b4b0162b53cb2d638417695c16d0c42e7d640c41661047e8ebf58de972', 'stage2_extractor_used': 'markitdown'}
convenio_2_2013.pdf
  cleaned_text (6300 chars): 'Decreto No 143712013\nConvenio N" ,..,,,Q?.... _.- -...,...\n1 .k?. aal Folio -49.. Tomo ..,i ,...\n!i I\nSecretaria de Gobierno - Direccton Gral. de Gobie.\n. .\nCONVENIO DE PESTAMO DE US GRATUITO dwk%died'...
  entities: {'doc_type_hint': None, 'number': 'D

## 6 — Inspect the `ClassificationRecord` Stage 4 wrote

This is the full accumulated decision: the primary LLM classifier's label and
confidence, the real BETO Second Opinion Agent's own label and OOD/SVM signals,
smells and risk score, which `review_route` the confidence gate picked, whether the
LLM Judge tier ran, and where `RoutingNode` physically moved the file.

In [6]:
async def _find_classification_record(job_id: str) -> ClassificationRecord | None:
    async with session_factory() as session:
        result = await session.execute(
            select(ClassificationRecord).where(ClassificationRecord.job_id == job_id)
        )
        return result.scalar_one_or_none()


for _name, _job_id in job_ids.items():
    record = await _find_classification_record(_job_id)
    print(f"{_name}")
    if record is None:
        print("  no ClassificationRecord -- job wasn't accepted, or classification failed")
        continue
    print(f"  label                      : {record.label!r}")
    print(f"  confidence                 : {record.confidence:.3f}")
    print(f"  all_scores                 : {record.all_scores}")
    print(f"  second_opinion_label       : {record.second_opinion_label!r}")
    print(f"  second_opinion_confidence  : {record.second_opinion_confidence:.3f}")
    print(f"  classifier_disagreement    : {record.classifier_disagreement}")
    print(f"  svm_agrees_with_prediction : {record.svm_agrees_with_prediction}")
    print(f"  smells                     : {record.smells}")
    print(f"  risk_score                 : {record.risk_score}")
    print(f"  review_route               : {record.review_route!r}")
    print(f"  judged_by_llm              : {record.judged_by_llm}")
    print(f"  stored_path                : {record.stored_path!r}")
    print()

ordenanza_10000_2019.pdf
  label                      : 'ordenanzas'
  confidence                 : 0.900
  all_scores                 : {'ordenanzas': 0.9}
  second_opinion_label       : 'ordenanza'
  second_opinion_confidence  : 0.997
  classifier_disagreement    : False
  svm_agrees_with_prediction : True
  smells                     : []
  risk_score                 : 0
  review_route               : 'accept'
  judged_by_llm              : False
  stored_path                : 'C:\\Users\\leona\\source\\repos\\Trabajo-Integrador\\storage\\documents\\classified\\ordenanzas\\72f4b5bc-98ae-420b-82cd-5a8c117798c6_ordenanza_10000_2019.pdf'

convenio_2_2013.pdf
  label                      : 'convenios'
  confidence                 : 0.900
  all_scores                 : {'convenios': 0.9}
  second_opinion_label       : 'decretos_concejo_municipal'
  second_opinion_confidence  : 0.677
  classifier_disagreement    : False
  svm_agrees_with_prediction : True
  smells                     : []

## 7 — The human-review queue and a manual decision

`GET /classification/review-queue` lists every `ClassificationRecord` with
`review_route == "human_review"` — whatever the confidence gate and Second Opinion
Agent's real signals produced for these three documents. If nothing landed in
review (all three auto-accepted), this section still demonstrates the endpoint
shape; re-run with different/harder sample PDFs to see a non-empty queue.

If at least one job is in the queue, this section also submits a real human
decision via `POST /classification/{job_id}/decision` and confirms the record's
`review_route` flips to `"accept"` with `human_overridden=True`, and that the file
physically moves to `classified/<label>/`.

In [7]:
response = client.get("/classification/review-queue", headers=auth_headers)
queue = response.json()
print(f"{len(queue)} job(s) awaiting human review")
for item in queue:
    print(f"  {item['jobId']}: label={item['label']!r} smells={item['smells']}")

5 job(s) awaiting human review
  68331cd1-cbad-4052-9588-a7326745eacb: label='decreto_ordenanzas' smells=['classifier_disagreement']
  cd956223-bbe6-45bc-a0ab-8f61066f5064: label='decretos' smells=['classifier_disagreement']
  a62fe938-eda7-4a5b-8937-368275ac0b8e: label='decretos' smells=['classifier_disagreement']
  dc7b2862-d9a0-428e-8b15-a380f3f4c682: label='decretos' smells=['classifier_disagreement']
  b47a3d6d-e524-4738-9173-35cbb329e2ab: label='decretos' smells=['classifier_disagreement']


In [8]:
if queue:
    _decide_job_id = queue[0]["jobId"]
    response = client.post(
        f"/classification/{_decide_job_id}/decision",
        json={"label": "ordenanzas", "notes": "reviewed in playground notebook"},
        headers=auth_headers,
    )
    print(f"decision status: {response.status_code}")

    record = await _find_classification_record(_decide_job_id)
    assert record is not None
    print(f"review_route      : {record.review_route!r}")
    print(f"human_overridden  : {record.human_overridden}")
    print(f"stored_path       : {record.stored_path!r}")
else:
    print("nothing in the review queue this run -- skipping the decision demo")

2026-08-21 15:15:37.044 | INFO     | classiflow.services.audit.service:record:37 - audit | job=68331cd1-cbad-4052-9588-a7326745eacb node=classification_decision event=human_decision
2026-08-21 15:15:37.054 | INFO     | classiflow.services.audit.service:record:37 - audit | job=68331cd1-cbad-4052-9588-a7326745eacb node=classification_routing event=passed


decision status: 200
review_route      : 'accept'
human_overridden  : True
stored_path       : 'C:\\Users\\leona\\source\\repos\\Trabajo-Integrador\\storage\\documents\\classified\\ordenanzas\\68331cd1-cbad-4052-9588-a7326745eacb_boletin_2056_2026.pdf'


## 8 — Where the files physically ended up

`storage/documents/` mirrors every `ClassificationRecord.stored_path` above —
`classified/<label>/` for auto-accepted or human-decided documents,
`review/human_review/` for anything still awaiting a decision.

In [9]:
_storage_root = Path(Settings.document_storage_root)
if _storage_root.exists():
    for _path in sorted(_storage_root.rglob("*")):
        if _path.is_file():
            print(_path.relative_to(_storage_root))
else:
    print(f"{_storage_root} does not exist yet")

classified\convenios\a4f84798-7744-4f42-b2b4-de3eac75c0d9_convenio_229_2021.pdf
classified\convenios\ae5279c5-43a1-49f5-9655-310868a3bec1_convenio_2_2013.pdf
classified\convenios\cf04d7f9-72c1-4061-bb15-5f3565ebfbdf_convenio_10_2013.pdf
classified\declaraciones_concejo_municipal\711c526d-64c7-42d6-b244-c1d1f847fddd_declaracion_4918_2009.pdf
classified\decreto\036ce319-110d-4ac2-bbf6-51c60b936e70_declaracion_2501_1991.pdf
classified\decreto\2ae93185-96c7-43e1-b3f2-9de890c62574_decreto_cm_10554_1995.pdf
classified\decreto\42d3fe0c-e2b1-4987-9c28-d5ca2f3249a1_boletin_1000_2019.pdf
classified\decretos\612d5ab1-ee99-41da-a4af-3b1d2f949ee4_decreto_1001_2016.pdf
classified\decretos\a65f531b-a2ea-4300-9e74-ee5a9ba39032_decreto_1000_2008.pdf
classified\ordenanzas\68331cd1-cbad-4052-9588-a7326745eacb_boletin_2056_2026.pdf
classified\ordenanzas\72f4b5bc-98ae-420b-82cd-5a8c117798c6_ordenanza_10000_2019.pdf
classified\resoluciones\8b0222c1-0997-4207-97b9-bc20fbfeb582_resolucion_cm_197_2024.pdf
clas

## 9 — Accuracy summary: predicted vs. expected label

`_SAMPLE_FILES` carries the ground-truth label for every document (the labeled
corpus folder it was pulled from). This section compares that expected label
against what the pipeline actually produced, and separately reports:
- Any file that never reached classification -- either it crashed during ingest
  (`ingest_errors` from section 3, e.g. the JSON-escaping bug where the model
  echoes OCR-garbled text with a stray `"` into its own `reasoning` field,
  breaking `_extract()`'s JSON parsing) or it was correctly held earlier in the
  pipeline (e.g. node3's language detector rejecting a heavily OCR-degraded scan).
- For every wrong prediction, whether `classifier_disagreement` actually caught it
  (routed to `human_review`) or whether the primary classifier and BETO's Second
  Opinion Agent happened to agree on the same wrong label, letting it slip through
  to `accept` uncaught -- this is the real question behind "is the safety net
  working," not just "is the primary classifier accurate."

Builds one structured row per document (`run_rows`) so this same data drives both
the console summary below and section 10's HTML report -- no duplicated logic
between the two.

In [10]:
run_rows: list[dict[str, object]] = []

for _name, _expected in _SAMPLE_FILES.items():
    _row: dict[str, object] = {"filename": _name, "expected": _expected}

    if _name in ingest_errors:
        _row["outcome"] = "crashed"
        _row["detail"] = ingest_errors[_name]
        run_rows.append(_row)
        continue

    _job_id = job_ids.get(_name)
    _job = await _find_job(_job_id) if _job_id else None
    _record = await _find_classification_record(_job_id) if _job_id else None

    if _record is None:
        _row["outcome"] = "held_earlier"
        _row["detail"] = (
            f"{_job.failed_at_node}: {_job.rejection_reason}"
            if _job is not None and _job.failed_at_node
            else "no ClassificationRecord produced"
        )
        run_rows.append(_row)
        continue

    _is_correct = _record.label == _expected
    _row.update({
        "outcome": "correct" if _is_correct else "wrong",
        "predicted": _record.label,
        "confidence": _record.confidence,
        "second_opinion_label": _record.second_opinion_label,
        "second_opinion_confidence": _record.second_opinion_confidence,
        "disagreement": _record.classifier_disagreement,
        "review_route": _record.review_route,
        "smells": _record.smells,
        "caught_by_safety_net": (not _is_correct) and _record.review_route == "human_review",
    })
    run_rows.append(_row)

_correct = sum(1 for r in run_rows if r["outcome"] == "correct")
_wrong = [r for r in run_rows if r["outcome"] == "wrong"]
_wrong_caught = [r for r in _wrong if r["caught_by_safety_net"]]
_wrong_uncaught = [r for r in _wrong if not r["caught_by_safety_net"]]
_held_earlier = [r for r in run_rows if r["outcome"] == "held_earlier"]
_crashed = [r for r in run_rows if r["outcome"] == "crashed"]
_total = len(run_rows)

print(f"correct        : {_correct}/{_total}")
print(f"wrong, caught  : {len(_wrong_caught)}/{_total}  (disagreement -> human_review)")
print(f"wrong, uncaught: {len(_wrong_uncaught)}/{_total}  (both classifiers agreed, wrongly)")
print(f"held earlier   : {len(_held_earlier)}/{_total}  (never reached classification)")
print(f"crashed        : {len(_crashed)}/{_total}  (ingest raised uncaught)")
print()

if _wrong:
    print("Wrong predictions (filename -> expected / got, caught_by_safety_net):")
    for r in _wrong:
        expected, predicted = r["expected"], r["predicted"]
        caught = r["caught_by_safety_net"]
        print(f"  {r['filename']}: {expected!r} -> {predicted!r}, caught={caught}")
    print()

if _held_earlier:
    print("Held earlier in the pipeline (no ClassificationRecord):")
    for r in _held_earlier:
        print(f"  {r['filename']}: {r['detail']}")
    print()

if _crashed:
    print("Crashed during ingest:")
    for r in _crashed:
        print(f"  {r['filename']}: {r['detail']}")

correct        : 8/18
wrong, caught  : 4/18  (disagreement -> human_review)
wrong, uncaught: 5/18  (both classifiers agreed, wrongly)
held earlier   : 1/18  (never reached classification)
crashed        : 0/18  (ingest raised uncaught)

Wrong predictions (filename -> expected / got, caught_by_safety_net):
  boletin_2056_2026.pdf: 'boletines' -> 'ordenanzas', caught=False
  boletin_1000_2019.pdf: 'boletines' -> 'decreto', caught=False
  declaracion_2501_1991.pdf: 'declaraciones_concejo_municipal' -> 'decreto', caught=False
  decreto_ordenanza_1182_1976.pdf: 'decreto_ordenanzas' -> 'decretos', caught=True
  decreto_cm_1016_2025.pdf: 'decretos_concejo_municipal' -> 'decretos', caught=True
  decreto_cm_10554_1995.pdf: 'decretos_concejo_municipal' -> 'decreto', caught=False
  resolucion_102_2021.pdf: 'resoluciones' -> 'decretos', caught=True
  resolucion_cm_16879_2024.pdf: 'resoluciones_concejo_municipal' -> 'decretos', caught=True
  resolucion_cm_197_2024.pdf: 'resoluciones_concejo_municip

## 10 — HTML report

Renders `run_rows` (built in section 9) into a self-contained HTML file, one row
per document, plus the same stat strip and safety-net breakdown printed above.
Regenerates from scratch every run -- open the file this cell prints after each
notebook execution to see that run's own report, not a stale one.

In [12]:
import html
from datetime import datetime, timezone

_OUTCOME_PILL = {
    "correct": '<span class="pill good"><span class="dot"></span>{label}</span>',
    "wrong": '<span class="pill wrong">{label}</span>',
    "held_earlier": '<span class="pill crash">{label}</span>',
    "crashed": '<span class="pill crash">{label}</span>',
}


def _esc(value: object) -> str:
    return html.escape(str(value))


def _render_row(row: dict[str, object]) -> str:
    outcome = row["outcome"]
    filename = _esc(row["filename"])
    expected = _esc(row["expected"])

    if outcome in {"held_earlier", "crashed"}:
        label = "held earlier" if outcome == "held_earlier" else "crashed"
        predicted_cell = _OUTCOME_PILL[outcome].format(label=label)
        return (
            '\n          <tr class="row-crash">'
            f'\n            <td class="doc-name">{filename}</td>'
            f"\n            <td>{expected}</td>"
            f"\n            <td>{predicted_cell}</td>"
            '\n            <td class="conf">&mdash;</td>'
            '\n            <td class="doc-name">&mdash;</td>'
            '\n            <td><span class="pill neutral">n/a</span></td>'
            '\n            <td><span class="pill crash">review</span></td>'
            f'\n            <td class="smells">{_esc(row["detail"])}</td>'
            "\n          </tr>"
        )

    predicted = _esc(row["predicted"])
    predicted_cell = _OUTCOME_PILL[outcome].format(label=predicted)
    uncaught = outcome == "wrong" and not row["caught_by_safety_net"]
    row_class = "row-wrong-uncaught" if uncaught else ""
    if uncaught:
        disagreement_pill = '<span class="pill wrong">no &mdash; should have</span>'
    elif row["disagreement"]:
        disagreement_pill = '<span class="pill warn">yes</span>'
    else:
        disagreement_pill = '<span class="pill neutral">no</span>'
    route = _esc(row["review_route"])
    route_pill_class = "warn" if route == "human_review" else "good"
    smells = ", ".join(row["smells"]) if row["smells"] else "&mdash;"
    second_opinion = _esc(row["second_opinion_label"])
    second_conf = row["second_opinion_confidence"]

    return (
        f'\n          <tr class="{row_class}">'
        f'\n            <td class="doc-name">{filename}</td>'
        f"\n            <td>{expected}</td>"
        f"\n            <td>{predicted_cell}</td>"
        f'\n            <td class="conf">{row["confidence"]:.3f}</td>'
        f'\n            <td class="doc-name">{second_opinion} ({second_conf:.3f})</td>'
        f"\n            <td>{disagreement_pill}</td>"
        f'\n            <td><span class="pill {route_pill_class}">{route}</span></td>'
        f'\n            <td class="smells">{smells}</td>'
        "\n          </tr>"
    )


_rows_html = "\n".join(_render_row(r) for r in run_rows)

_generated_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
_categories_covered = len(set(_SAMPLE_FILES.values()))

_intro_text = (
    f"{_total} municipal documents, drawn from the labeled corpus across "
    f"{_categories_covered} categories, run through the real ingestion &rarr; "
    "extraction &rarr; enrichment &rarr; classification pipeline &mdash; real "
    "Phi-4-mini primary classifier, real BETO v2 second opinion, real "
    "confidence gate and routing."
)
_table_intro_text = (
    "Expected label comes from the labeled corpus folder each file was pulled "
    "from. Rows shaded red are wrong predictions the safety net did "
    "<em>not</em> catch &mdash; both classifiers agreed on the same wrong "
    "answer. Rows shaded purple never reached classification."
)
_caught_finding_text = (
    f"{len(_wrong_caught)} of {len(_wrong)} wrong predictions were caught "
    "&mdash; <code>classifier_disagreement</code> fired because the primary "
    "classifier and BETO v2's second opinion landed on different labels, "
    "routing the document to <code>human_review</code> instead of silently "
    "auto-accepting a wrong answer."
)
_uncaught_finding_text = (
    f"{len(_wrong_uncaught)} of {len(_wrong)} wrong predictions were "
    "<em>not</em> caught &mdash; both classifiers agreed on the same wrong "
    "label, which is exactly the signal the system trusts. Agreement "
    "between two independent models isn't proof of correctness on its own."
)
_held_finding_text = (
    f"{len(_held_earlier)} document(s) never reached classification at all "
    "&mdash; caught by an earlier gate (e.g. node3's language/content "
    "validation) rather than by classification-stage logic. See the "
    'table\'s "Smells / detail" column for the specific reason.'
)

_reports_dir = _project_root / "storage" / "reports"
_reports_dir.mkdir(parents=True, exist_ok=True)
_report_stamp = f"{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
_report_path = _reports_dir / f"classification_report_{_report_stamp}.html"

_template_path = Path(classiflow.__file__).parent / "playground" / "stage4" / "report_template.html"
_report_html = _template_path.read_text(encoding="utf-8")

_substitutions = {
    "__INTRO_TEXT__": _intro_text,
    "__GENERATED_AT__": _generated_at,
    "__TOTAL__": str(_total),
    "__CORRECT__": str(_correct),
    "__WRONG_CAUGHT__": str(len(_wrong_caught)),
    "__WRONG_UNCAUGHT__": str(len(_wrong_uncaught)),
    "__HELD_EARLIER__": str(len(_held_earlier)),
    "__CRASHED__": str(len(_crashed)),
    "__TABLE_INTRO_TEXT__": _table_intro_text,
    "__ROWS_HTML__": _rows_html,
    "__CAUGHT_FINDING_TEXT__": _caught_finding_text,
    "__UNCAUGHT_FINDING_TEXT__": _uncaught_finding_text,
    "__HELD_FINDING_TEXT__": _held_finding_text,
}
for _token, _value in _substitutions.items():
    _report_html = _report_html.replace(_token, _value)

_report_path.write_text(_report_html, encoding="utf-8")
print(f"report written to {_report_path}")

report written to C:\Users\leona\source\repos\Trabajo-Integrador\storage\reports\classification_report_20260821_182009.html
